# Conventions Qiskit ↔ Pulser

Expériences qui justifient chaque ligne de `formalisations/conventions/conventions.typ`.

Cadre : `DigitalAnalogDevice`, canal `raman_local`, base `"digital"`.

In [1]:
import warnings

import numpy as np
from pulser import Pulse, Register, Sequence
from pulser.devices import DigitalAnalogDevice
from pulser.waveforms import BlackmanWaveform
from pulser_simulation import QutipEmulator
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

warnings.filterwarnings("ignore", category=DeprecationWarning)


def pulse(area, phase=0.0):
    return Pulse.ConstantDetuning(
        BlackmanWaveform(duration=500, area=area), detuning=0, phase=phase
    )


def sequence(n_qubits=1, spacing=20.0):
    """Atomes q0, q1, ... alignés, canal `raman` (raman_local) ciblant q0."""
    reg = Register({f"q{i}": (i * spacing, 0) for i in range(n_qubits)})
    seq = Sequence(reg, DigitalAnalogDevice)
    seq.declare_channel("raman", "raman_local", initial_target="q0")
    return seq


def run(seq, shots=1000):
    """Renvoie (statevector final, comptage des bitstrings)."""
    seq.measure(basis="digital")
    res = QutipEmulator.from_sequence(seq).run()
    psi = res.get_final_state().full().flatten()
    return psi, dict(res.sample_final_state(N_samples=shots))


def qiskit_sv(qc):
    return Statevector.from_instruction(qc).data


def global_phase(a, b):
    """λ tel que a = λ·b si a et b ne diffèrent que d'une phase globale, sinon None."""
    k = np.argmax(np.abs(b))
    lam = a[k] / b[k]
    return lam if np.allclose(a, lam * b, atol=1e-3) else None

## 1. Correspondance des états

Une impulsion π doit amener |0⟩ sur |1⟩, une impulsion 2π doit le ramener sur |0⟩ (à une phase −1 près).

On ne peut pas simuler « aucune impulsion » en base `digital` : Pulser déduit la base de simulation des impulsions non nulles et retombe sinon sur `ground-rydberg`.

In [2]:
for area in (np.pi, 2 * np.pi):
    seq = sequence()
    seq.add(pulse(area), "raman")
    psi, counts = run(seq)
    print(
        f"aire {area / np.pi:.0f}π → statevector",
        np.round(psi, 3),
        "| bitstrings",
        counts,
    )

aire 1π → statevector [0.-0.j 1.+0.j] | bitstrings {'1': 1000}
aire 2π → statevector [1.+0.j 0.-0.j] | bitstrings {'0': 1000}


## 2. Aire → angle

Une rotation d'angle θ depuis |0⟩ donne P(1) = sin²(θ/2). On compare avec l'aire de l'impulsion.

In [3]:
for area in (np.pi / 4, np.pi / 2, 2 * np.pi / 3, np.pi, 3 * np.pi / 2):
    seq = sequence()
    seq.add(pulse(area), "raman")
    psi, _ = run(seq)
    print(
        f"aire {area:.3f} : P(1) mesurée {abs(psi[1]) ** 2:.3f}"
        f" | sin²(aire/2) = {np.sin(area / 2) ** 2:.3f}"
    )

aire 0.785 : P(1) mesurée 0.146 | sin²(aire/2) = 0.146
aire 1.571 : P(1) mesurée 0.500 | sin²(aire/2) = 0.500
aire 2.094 : P(1) mesurée 0.750 | sin²(aire/2) = 0.750


aire 3.142 : P(1) mesurée 1.000 | sin²(aire/2) = 1.000
aire 4.712 : P(1) mesurée 0.500 | sin²(aire/2) = 0.500


## 3. Impulsion de phase 0 et impulsion de phase π/2

On compare le statevector Pulser aux candidats Qiskit, à une phase globale près.

In [4]:
theta = np.pi / 2
for phase in (0.0, np.pi / 2):
    seq = sequence()
    seq.add(pulse(theta, phase), "raman")
    psi, _ = run(seq)
    print(f"phase {phase:.3f} : statevector {np.round(psi, 3)}")
    for name, gate in (("rx", "rx"), ("ry", "ry")):
        for sign in (+1, -1):
            qc = QuantumCircuit(1)
            getattr(qc, gate)(sign * theta, 0)
            lam = global_phase(psi, qiskit_sv(qc))
            if lam is not None:
                print(
                    f"   = {np.round(lam, 3)} · {name}({'+' if sign > 0 else '-'}θ)|0⟩"
                )

phase 0.000 : statevector [0.707+0.j    0.   -0.707j]
   = (1+0j) · rx(+θ)|0⟩
phase 1.571 : statevector [ 0.707+0.j -0.707+0.j]
   = (1+0j) · ry(-θ)|0⟩


## 4. `phase_shift(φ)` : Rz(+φ) ou Rz(−φ) ?

Impulsion π/2 de phase 0 (état sur −y), `phase_shift(π/2)`, puis impulsion π/2 de phase π/2 (rotation autour de y).
Rz(+π/2) et Rz(−π/2) envoient −y sur +x et sur −x, que la dernière impulsion amène sur des pôles opposés : les **populations** tranchent.

On ne compare pas les statevectors : le virtual Z ne déphase que les impulsions suivantes, l'état Pulser diffère donc de l'état Qiskit d'un Rz final.

In [5]:
seq = sequence()
seq.add(pulse(np.pi / 2, phase=0), "raman")
seq.phase_shift(np.pi / 2, "q0", basis="digital")
seq.add(pulse(np.pi / 2, phase=np.pi / 2), "raman")
psi, counts = run(seq)
print(
    "Pulser             : populations",
    np.round(np.abs(psi) ** 2, 3),
    "| bitstrings",
    counts,
)

# phase 0 = rx(+θ), phase π/2 = ry(-θ) (cf. section 3)
for sign in (+1, -1):
    qc = QuantumCircuit(1)
    qc.rx(np.pi / 2, 0)
    qc.rz(sign * np.pi / 2, 0)
    qc.ry(-np.pi / 2, 0)
    probs = Statevector.from_instruction(qc).probabilities()
    print(
        f"Qiskit rz({'+' if sign > 0 else '-'}π/2) : populations {np.round(probs, 3)}"
    )

Pulser             : populations [1. 0.] | bitstrings {'0': 1000}
Qiskit rz(+π/2) : populations [1. 0.]
Qiskit rz(-π/2) : populations [0. 1.]


## 5. Ordre des bits avec 2 atomes

Deux atomes éloignés, impulsion π sur **un seul** d'entre eux. Qiskit est little-endian : `q0` est le bit le plus à droite, `X` sur `q0` donne `'01'`.

In [6]:
for target in ("q0", "q1"):
    seq = sequence(n_qubits=2)
    seq.target(target, "raman")
    seq.add(pulse(np.pi), "raman")
    psi, counts = run(seq)

    qc = QuantumCircuit(2)
    qc.x(int(target[1]))
    sv = Statevector.from_instruction(qc)

    print(f"π sur {target}")
    print("   Pulser : statevector", np.round(psi, 3), "| bitstrings", counts)
    print(
        "   Qiskit : statevector",
        np.round(sv.data, 3),
        "| bitstrings",
        {str(k): int(v) for k, v in sv.sample_counts(1000).items()},
    )

π sur q0
   Pulser : statevector [0.-0.j 0.+0.j 1.+0.j 0.+0.j] | bitstrings {'10': 1000}
   Qiskit : statevector [0.+0.j 1.+0.j 0.+0.j 0.+0.j] | bitstrings {'01': 1000}
π sur q1
   Pulser : statevector [0.-0.j 1.+0.j 0.+0.j 0.+0.j] | bitstrings {'01': 1000}
   Qiskit : statevector [0.+0.j 0.+0.j 1.+0.j 0.+0.j] | bitstrings {'10': 1000}
